### Search Engine With Tools And Agents

In [8]:
## Arxiv--Research
## Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [9]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
print(wiki.name)

wikipedia


In [10]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [11]:
tools=[wiki,arxiv]

In [12]:
'''## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Added HuggingFace embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings'''

# Example usage with HuggingFace embeddings:
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


'## Custom tools[RAG Tool]\nfrom langchain_community.document_loaders import WebBaseLoader\nfrom langchain_community.vectorstores import FAISS\nfrom langchain_openai import OpenAIEmbeddings\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter\n# Added HuggingFace embeddings\nfrom langchain_community.embeddings import HuggingFaceEmbeddings'

In [13]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
# Example of a tiny model (might be less accurate)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-MiniLM-L3-v2")


# Initialize with an empty list or a very small batch
vectordb = FAISS.from_documents(documents[:1], embeddings)

# Loop through the rest in batches (e.g., batch_size of 10)
batch_size = 10
for i in range(1, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    if batch:
        print(f"Processing batch {i} to {i + len(batch)}...")
        vectordb.add_documents(batch)

retriever = vectordb.as_retriever(search_kwargs={"k": 3})
print("✅ RAG setup complete!")


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing batch 1 to 11...
Processing batch 11 to 21...
Processing batch 21 to 31...
Processing batch 31 to 41...
Processing batch 41 to 51...
Processing batch 51 to 61...
Processing batch 61 to 66...
✅ RAG setup complete!


In [1]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain.agents import create_openai_tools_agent, AgentExecutor  
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = HuggingFaceEndpoint(repo_id="google/flan-t5-large", temperature=0.7)
tools = [retriever_tool]
prompt = ChatPromptTemplate.from_messages([
    ("system", "Helpful assistant"),
    MessagesPlaceholder("agent_scratchpad"),
    ("human", "{input}")
])

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


ImportError: cannot import name 'create_openai_tools_agent' from 'langchain.agents' (c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain\agents\__init__.py)

In [ ]:
tools=[wiki,arxiv,retriever_tool]

In [ ]:
tools

In [ ]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import openai
load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")
openai.api_key=os.getenv("OPENAI_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-8b-instant")

In [ ]:
## Prompt Template
from langchain import hub
prompt=hub.pull("hwchase17/openai-functions-agent")
prompt.messages

ImportError: cannot import name 'hub' from 'langchain' (c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain\__init__.py)

In [ ]:
'''## Agents
from langchain.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)
agent'''

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. FREE HuggingFace LLM (costs 0 credits first 1M tokens/day)
llm = HuggingFaceEndpoint(
    repo_id="microsoft/DialoGPT-medium",  # or "google/flan-t5-large"
    temperature=0.7,
    max_new_tokens=512,
    huggingfacehub_api_token="your_hf_token_here"  # Free from huggingface.co/settings/tokens
)

# 2. Your tools (from earlier RAG + retriever_tool)
tools = [retriever_tool]  # Add more tools

# 3. Agent prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful research assistant. Use tools to answer."),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
    ("human", "{input}")
])

# 4. Create agent (same function works!)
agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 5. Test
response = agent_executor.invoke({"input": "What is Langsmith?"})
print(response['output'])


In [ ]:
## Agent Executer
from langchain.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

In [ ]:
agent_executor.invoke({"input":"Tell me about Langsmith"})

In [ ]:
agent_executor.invoke({"input":"What is machine learning"})

In [ ]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})